# Base Wrappers

> Non-3rd party wrappers for the environments. These are used to modify the behavior of the environments, such as changing the observation space, action space, or reward structure.

In [ ]:
#| default_exp wrappers.base

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from __future__ import annotations

import gymnasium as gym
import numba as nb
import numpy as np

from gymnasium import spaces
from gymnasium.core import ObservationWrapper
from numpy.typing import NDArray as ndarray

from multigrid.envs.base import MultiGridEnv, AgentID, ObsType
from multigrid.core.constants import Color, Direction, State, Type
from multigrid.core.world_object import WorldObj


In [ ]:
#| export

class FullyObsWrapper(ObservationWrapper):
    """
    Fully observable gridworld using a compact grid encoding instead of agent view.

    Examples
    --------
    >>> import gymnasium as gym
    >>> import multigrid.envs
    >>> env = gym.make('MultiGrid-Empty-16x16-v0')
    >>> obs, _ = env.reset()
    >>> obs[0]['image'].shape
    (7, 7, 3)

    >>> from multigrid.wrappers import FullyObsWrapper
    >>> env = FullyObsWrapper(env)
    >>> obs, _ = env.reset()
    >>> obs[0]['image'].shape
    (16, 16, 3)
    """

    def __init__(self, env: MultiGridEnv):
        """
        """
        super().__init__(env)

        # Update agent observation spaces
        for agent in self.env.agents:
            agent.observation_space['image'] = spaces.Box(
                low=0, high=255, shape=(env.height, env.width, WorldObj.dim), dtype=int)

    def observation(self, obs: dict[AgentID, ObsType]) -> dict[AgentID, ObsType]:
        """
        :meta private:
        """
        img = self.env.grid.encode()
        for agent in self.env.agents:
            img[agent.state.pos] = agent.encode()

        for agent_id in obs:
            obs[agent_id]['image'] = img

        return obs



In [ ]:
#| export

class ImgObsWrapper(ObservationWrapper):
    """
    Use the image as the only observation output for each agent.

    Examples
    --------
    >>> import gymnasium as gym
    >>> import multigrid.envs
    >>> env = gym.make('MultiGrid-Empty-8x8-v0')
    >>> obs, _ = env.reset()
    >>> obs[0].keys()
    dict_keys(['image', 'direction', 'mission'])

    >>> from multigrid.wrappers import ImgObsWrapper
    >>> env = ImgObsWrapper(env)
    >>> obs, _ = env.reset()
    >>> obs.shape
    (7, 7, 3)
    """

    def __init__(self, env: MultiGridEnv):
        """
        """
        super().__init__(env)

        # Update agent observation spaces
        for agent in self.env.agents:
            agent.observation_space = agent.observation_space['image']
            agent.observation_space.dtype = np.uint8

    def observation(self, obs: dict[AgentID, ObsType]) -> dict[AgentID, ObsType]:
        """
        :meta private:
        """
        for agent_id in obs:
            obs[agent_id] = obs[agent_id]['image'].astype(np.uint8)

        return obs



In [ ]:
#| export

class OneHotObsWrapper(ObservationWrapper):
    """
    Wrapper to get a one-hot encoding of a partially observable
    agent view as observation.

    Examples
    --------
    >>> import gymnasium as gym
    >>> import multigrid.envs
    >>> env = gym.make('MultiGrid-Empty-5x5-v0')
    >>> obs, _ = env.reset()
    >>> obs[0]['image'][0, :, :]
    array([[2, 5, 0],
            [2, 5, 0],
            [2, 5, 0],
            [2, 5, 0],
            [2, 5, 0],
            [2, 5, 0],
            [2, 5, 0]])

    >>> from multigrid.wrappers import OneHotObsWrapper
    >>> env = OneHotObsWrapper(env)
    >>> obs, _ = env.reset()
    >>> obs[0]['image'][0, :, :]
    array([[0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0],
            [0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0],
            [0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0],
            [0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0],
            [0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0],
            [0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0],
            [0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0]],
            dtype=uint8)
    """

    def __init__(self, env: MultiGridEnv):
        """
        """
        super().__init__(env)
        self.dim_sizes = np.array([
            len(Type), len(Color), max(len(State), len(Direction))])

        # Update agent observation spaces
        dim = sum(self.dim_sizes)
        for agent in self.env.agents:
            view_height, view_width, _ = agent.observation_space['image'].shape
            agent.observation_space['image'] = spaces.Box(
                low=0, high=1, shape=(view_height, view_width, dim), dtype=np.uint8)

    def observation(self, obs: dict[AgentID, ObsType]) -> dict[AgentID, ObsType]:
        """
        :meta private:
        """
        for agent_id in obs:
            obs[agent_id]['image'] = self.one_hot(obs[agent_id]['image'], self.dim_sizes)

        return obs

    @staticmethod
    @nb.njit(cache=True)
    def one_hot(x: ndarray[np.int64], dim_sizes: ndarray[np.int64]) -> ndarray[np.uint8]:
        """
        Return a one-hot encoding of a 3D integer array,
        where each 2D slice is encoded separately.

        Parameters
        ----------
        x : ndarray[int] of shape (view_height, view_width, dim)
            3D array of integers to be one-hot encoded
        dim_sizes : ndarray[int] of shape (dim,)
            Number of possible values for each dimension

        Returns
        -------
        out : ndarray[uint8] of shape (view_height, view_width, sum(dim_sizes))
            One-hot encoding

        :meta private:
        """
        out = np.zeros((x.shape[0], x.shape[1], sum(dim_sizes)), dtype=np.uint8)

        dim_offset = 0
        for d in range(len(dim_sizes)):
            for i in range(x.shape[0]):
                for j in range(x.shape[1]):
                    k = dim_offset + x[i, j, d]
                    out[i, j, k] = 1

            dim_offset += dim_sizes[d]

        return out



In [ ]:
#| export

class SingleAgentWrapper(gym.Wrapper):
    """
    Wrapper to convert a multi-agent environment into a
    single-agent environment.

    Examples
    --------
    >>> import gymnasium as gym
    >>> import multigrid.envs
    >>> env = gym.make('MultiGrid-Empty-5x5-v0')
    >>> obs, _ = env.reset()
    >>> obs[0].keys()
    dict_keys(['image', 'direction', 'mission'])

    >>> from multigrid.wrappers import SingleAgentWrapper
    >>> env = SingleAgentWrapper(env)
    >>> obs, _ = env.reset()
    >>> obs.keys()
    dict_keys(['image', 'direction', 'mission'])
    """

    def __init__(self, env: MultiGridEnv):
        """
        """
        super().__init__(env)
        self.observation_space = env.agents[0].observation_space
        self.action_space = env.agents[0].action_space

    def reset(self, *args, **kwargs):
        """
        :meta private:
        """
        result = super().reset(*args, **kwargs)
        return tuple(item[0] for item in result)

    def step(self, action):
        """
        :meta private:
        """
        result = super().step({0: action})
        return tuple(item[0] for item in result)


In [ ]:
#| export
import gymnasium as gym
import numpy as np
import os
import tqdm

def export_video(X, outfile, fps=30, rescale_factor=2):

    try:
        import moviepy.editor as mpy
    except:
        raise ImportError(
            "GridRecorder requires moviepy library. Try installing:\n $ pip install moviepy"
        )

    if isinstance(X, list):
        X = np.stack(X)

    if isinstance(X, float) and X.max() < 1:
        X = (X * 255).astype(np.uint8).clip(0, 255)
    print(X.shape)
    if rescale_factor is not None and rescale_factor != 1:
        print('`rescale_factor` is deprecated in `export_video`. ' +\
              'use `env.video_scale` instead. ')

    def make_frame(i):
        out = X[i]
        return out

    getframe = lambda t: make_frame(min(int(t * fps), len(X) - 1))
    clip = mpy.VideoClip(getframe, duration=len(X) / fps)

    outfile = os.path.abspath(os.path.expanduser(outfile))
    if not os.path.isdir(os.path.dirname(outfile)):
        os.makedirs(os.path.dirname(outfile))
    clip.write_videofile(outfile, fps=fps)


def render_frames(X, path, ext="png"):
    try:
        from PIL import Image
    except ImportError as e:
        raise ImportError(
            "Error importing from PIL in export_frames. Try installing PIL:\n $ pip install Pillow"
        )

    # If the path has a file extension, dump frames in a new directory with = path minus extension
    if "." in os.path.basename(path):
        path = os.path.splitext(path)[0]
    if not os.path.isdir(path):
        os.makedirs(path)

    for k, frame in tqdm.tqdm(enumerate(X), total=len(X)):
        Image.fromarray(frame, "RGB").save(os.path.join(path, f"frame_{k}.{ext}"))


In [ ]:
#| export
import cv2
class GridRecorder(gym.core.Wrapper):
    default_max_len = 1000
    default_video_kwargs = {
        'fps': 20,
        'rescale_factor': 1,
    }
    def __init__(
            self,
            env,
            save_root,
            max_steps=1000,
            auto_save_images=True,
            auto_save_videos=True,
            auto_save_interval=None,
            render_kwargs={},
            video_kwargs={}
            ):
        super().__init__(env)
        self.agents = env.agents
        self.get_goal_state = env.get_goal_state
        self.get_layout = env.get_layout
        self.frames = None
        self.ptr = 0
        self.reset_count = 0
        self.last_save = -10000
        self.recording = False
        self.save_root = self.fix_path(save_root)
        self.auto_save_videos = auto_save_videos
        self.auto_save_images = auto_save_images
        self.auto_save_interval = auto_save_interval
        self.render_kwargs = render_kwargs
        self.video_kwargs = {**self.default_video_kwargs, **video_kwargs}
        self.n_parallel = getattr(env, 'num_envs', 1)

        self.video_scale = 8 # default setting to see text clearly
        self.render_reward = True

        if max_steps is None:

            if hasattr(env, "max_steps") and env.max_steps != 0:
                self.max_steps = env.max_steps + 1
            else:
                self.max_steps = self.default_max_len + 1
        else:
            self.max_steps = max_steps + 1

    @staticmethod
    def fix_path(path):
        return os.path.abspath(os.path.expanduser(path))

    @property
    def unwrapped_env(self):
        env = self.env
        while isinstance(env, gym.Wrapper):
            env = env.env
        return env

    @property
    def should_record(self):
        if self.recording:
            return True
        if self.auto_save_interval is None:
            return False
        return (self.reset_count - self.last_save) >= self.auto_save_interval

    def export_frames(self,  episode_id=None, save_root=None):
        if save_root is None:
            save_root = self.save_root
        if episode_id is None:
            episode_id = f'frames_{self.reset_count}'
        render_frames(self.frames[:self.ptr], os.path.join(self.fix_path(save_root), episode_id))

    def export_video(self, episode_id=None, save_root=None):
        if save_root is None:
            save_root = self.save_root
        if episode_id is None:
            episode_id = f'video_{self.reset_count}.mp4'
        export_video(self.frames[:self.ptr],
                     os.path.join(self.fix_path(save_root), episode_id),
                     **self.video_kwargs)

    def export_both(self, episode_id, save_root=None):
        self.export_frames(f'{episode_id}_frames', save_root=save_root)
        self.export_video(f'{episode_id}.mp4', save_root=save_root)

    def reset(self, **kwargs):
        if self.should_record and self.ptr>0:
            self.append_current_frame()
            if self.auto_save_images:
                self.export_frames()
            if self.auto_save_videos:
                self.export_video()
            self.last_save = self.reset_count
        del self.frames
        self.frames = None
        self.ptr = 0
        self.reset_count += self.n_parallel
        return self.env.reset(**kwargs)

    def append_current_frame(self, reward_dict=None, info_dict=None):
        if self.should_record:
            new_frame = self.env.render()

            if isinstance(new_frame, list) or len(new_frame.shape) > 3:
                new_frame = new_frame[0]

            if self.video_scale != 1:
                new_frame = cv2.resize(new_frame, None,
                                       fx=self.video_scale,
                                       fy=self.video_scale,
                                       interpolation=cv2.INTER_AREA)

            # if self.render_reward and reward_dict is not None:
            #     to_render = ['rew',
            #                  *[f'{k[0]+k[-1]}: {v:.3f}' for k, v in
            #                    reward_dict.items()]
            #                  ]
            #     str_spacing = 30
            #     for i, text_to_render in enumerate(to_render):
            #         cv2.putText(new_frame, text_to_render,
            #                     (int(0.8 * new_frame.shape[1]),
            #                      int(0.1 * new_frame.shape[0]) + (i * str_spacing)),
            #                     cv2.FONT_HERSHEY_SIMPLEX, 1.0, (255, 255, 255), 2)

            if self.frames is None:
                self.frames = np.zeros(
                    (self.max_steps, *new_frame.shape), dtype=new_frame.dtype
                )

            self.frames[self.ptr] = new_frame
            self.ptr += 1
            # if self.ptr < len(self.frames):
            #     self.frames[self.ptr] = new_frame
            # self.ptr += 1

    def step(self, action):
        if self.ptr == 0:
            self.append_current_frame()
        obs, rewards, terminations, truncations, info = self.env.step(action)
        done = all(terminations.values()) or all(truncations.values())
        self.append_current_frame(reward_dict=rewards, info_dict=info)
        return obs, rewards, terminations, truncations, info


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()